In [1]:
import socket, os, torch, subprocess
print("hostname:", socket.gethostname())
print("pid:", os.getpid())
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("cuda device count:", torch.cuda.device_count())
    print("current device:", torch.cuda.current_device())
    subprocess.run(["nvidia-smi", "-L"], check=False)

hostname: gilbreth-d002.rcac.purdue.edu
pid: 96596
cuda available: True
cuda device count: 1
current device: 0
GPU 0: NVIDIA A30 (UUID: GPU-9f803576-0185-fea0-daee-42754e4aef4c)


In [2]:
import os
os.chdir("/home/zoellner/src/guided-diffusion")
print("cwd:", os.getcwd())

cwd: /home/zoellner/src/guided-diffusion


In [3]:
from datetime import datetime
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

In [4]:
def _find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "outputs").exists() and (candidate / "experiments").exists():
            return candidate
    raise FileNotFoundError(f"Could not find repo root from: {start}")


REPO_ROOT = _find_repo_root(Path.cwd())
for path in (REPO_ROOT / "experiments", REPO_ROOT / "guidance", REPO_ROOT):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import robosuite
robosuite_pkg_root = REPO_ROOT / "robosuite" / "robosuite"
robosuite.__file__ = str(robosuite_pkg_root / "__init__.py")
robosuite.__path__ = [str(robosuite_pkg_root)]
from robosuite.environments.base import make as robosuite_make
from robosuite.environments.manipulation.pick_place import PickPlaceCan

if not hasattr(robosuite, "make"):
    robosuite.make = robosuite_make

from robomimic.utils import torch_utils as TorchUtils
from diffusion_comparison import (
    DEFAULT_WORLD_MODEL_RUN_PATH,
    MethodSpec,
    _ensure_dir,
    _extract_current_obs_dict,
    _rotation_6d_to_rzz_torch,
    _run_rollout_episode,
    _save_rollout_artifacts,
    _summarize_rollout,
    load_policy_and_env,
)
from world_model_utils import build_state_from_obs_dict, load_init_state_json, load_model_for_eval


INIT_STATE_DIR = REPO_ROOT / "outputs/can_rollouts/snr/n16h300r60g1bYfYv1"
INIT_STATE_PATH = INIT_STATE_DIR / "init_state.json"
SEED = 7
GUIDANCE_SCALE = 25.0
HORIZON = 300
GUIDANCE_ROLLOUT_STEPS = 8
POLICY_CKPT_PATH = Path("/scratch/gilbreth/zoellner/diffusion_runs/can_mh_lowdim/20260209180642/models/model_epoch_2000.pth")

if not INIT_STATE_PATH.exists():
    raise FileNotFoundError(f"Missing init state: {INIT_STATE_PATH}")
if not POLICY_CKPT_PATH.exists():
    raise FileNotFoundError(f"Missing policy checkpoint: {POLICY_CKPT_PATH}")


def _make_eef_guidance_helpers(predictor: torch.nn.Module, stats: dict, device: str, rollout_steps: int = 8):
    state_mean = torch.tensor(stats["state_mean"], device=device, dtype=torch.float32).unsqueeze(0)
    state_std = torch.tensor(stats["state_std"], device=device, dtype=torch.float32).unsqueeze(0)
    action_mean = torch.tensor(stats["action_mean"], device=device, dtype=torch.float32).unsqueeze(0)
    action_std = torch.tensor(stats["action_std"], device=device, dtype=torch.float32).unsqueeze(0)
    delta_mean = torch.tensor(stats["delta_mean"], device=device, dtype=torch.float32).unsqueeze(0)
    delta_std = torch.tensor(stats["delta_std"], device=device, dtype=torch.float32).unsqueeze(0)

    def rollout_score_from_state(state_now: np.ndarray, actions: torch.Tensor) -> torch.Tensor:
        state = torch.as_tensor(state_now, device=device, dtype=torch.float32).unsqueeze(0)
        state = state.expand(actions.shape[0], -1).contiguous()

        horizon = min(int(rollout_steps), int(actions.shape[1]))
        eef_rzz_traj = []

        for step_idx in range(horizon):
            action_t = actions[:, step_idx, :]
            state_n = (state - state_mean) / state_std
            action_n = (action_t - action_mean) / action_std
            delta_n = predictor(state_n, action_n)
            delta = delta_n * delta_std + delta_mean
            state = state + delta
            eef_rzz_traj.append(_rotation_6d_to_rzz_torch(state[:, 21:27]))

        eef_rzz_stack = torch.stack(eef_rzz_traj, dim=1)
        return -eef_rzz_stack.mean(dim=1)

    def guidance_function(obs_dict: dict, actions: torch.Tensor) -> torch.Tensor:
        current_obs = _extract_current_obs_dict(obs_dict)
        state_now = build_state_from_obs_dict(current_obs)
        objective = rollout_score_from_state(state_now, actions).mean()
        return torch.autograd.grad(objective, actions, retain_graph=False, create_graph=False)[0]

    return rollout_score_from_state, guidance_function


def _sanitize_action(action: np.ndarray) -> np.ndarray:
    action = np.asarray(action, dtype=np.float32)
    action = np.nan_to_num(action, nan=0.0, posinf=0.0, neginf=0.0)
    return np.clip(action, -1.0, 1.0)


print(f"repo root: {REPO_ROOT}")
print(f"init state: {INIT_STATE_PATH}")
print(f"policy checkpoint: {POLICY_CKPT_PATH}")

[robosuite WARNING] No private macro file found! (macros.py:53)
[robosuite WARNING] It is recommended to use a private macro file (macros.py:54)
[robosuite WARNING] To setup, run: python /home/zoellner/src/guided-diffusion/robosuite/robosuite/scripts/setup_macros.py (macros.py:55)
/home/zoellner/.conda/envs/guided_diffusion/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


repo root: /home/zoellner/src/guided-diffusion
init state: /home/zoellner/src/guided-diffusion/outputs/can_rollouts/snr/n16h300r60g1bYfYv1/init_state.json
policy checkpoint: /scratch/gilbreth/zoellner/diffusion_runs/can_mh_lowdim/20260209180642/models/model_epoch_2000.pth


In [5]:
device = TorchUtils.get_torch_device(try_to_use_cuda=True)
env, policy, _ = load_policy_and_env(str(POLICY_CKPT_PATH), device=device, record_video=True)
predictor, stats, _, _ = load_model_for_eval(
    model_or_run_path=DEFAULT_WORLD_MODEL_RUN_PATH,
    predictor_kind="learned",
    device=device,
    load_val_trajectories=False,
)
score_fn, guidance_fn = _make_eef_guidance_helpers(
    predictor=predictor,
    stats=stats,
    device=device,
    rollout_steps=GUIDANCE_ROLLOUT_STEPS,
)

init_state = load_init_state_json(str(INIT_STATE_PATH))
print(f"loaded env + policy on device: {device}")
print(f"init state keys: {list(init_state.keys())[:5]}")


============= Initialized Observation Utils with Obs Spec =============

using obs modality: low_dim with keys: ['robot0_eef_quat', 'object', 'robot0_gripper_qpos', 'robot0_eef_pos']
using obs modality: rgb with keys: []
using obs modality: depth with keys: []
using obs modality: scan with keys: []
Created environment with name PickPlaceCan
Action size is 7
============= Loaded Config =============
{
    "algo_name": "diffusion_policy",
    "experiment": {
        "name": "can_mh_lowdim",
        "validate": false,
        "logging": {
            "terminal_output_to_txt": true,
            "log_tb": true,
            "log_wandb": true,
            "wandb_proj_name": "guided_diffusion"
        },
        "save": {
            "enabled": true,
            "every_n_seconds": null,
            "every_n_epochs": 50,
            "epochs": [],
            "on_best_validation": false,
            "on_best_rollout_return": false,
            "on_best_rollout_success_rate": true
        },
   

In [ ]:
GUIDANCE_SCALE = 30
method = MethodSpec(
    slug=f"dp_guidance_l{int(GUIDANCE_SCALE)}",
    label=f"Guidance λ={int(GUIDANCE_SCALE)}",
    kind="guided",
    guidance_scale=GUIDANCE_SCALE,
)

# Keep output directory creation in rollout cell.
run_root_base = REPO_ROOT / "outputs/can_rollouts/notebook_junk/single_rollouts"
run_id = f"{INIT_STATE_DIR.name}_{method.slug}_seed{SEED}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
run_output_path = _ensure_dir(run_root_base / run_id)
video_path = run_output_path / "video.mp4"

rollout = _run_rollout_episode(
    env=env,
    policy=policy,
    init_state=init_state,
    seed=SEED,
    horizon=HORIZON,
    video_path=video_path,
    camera_names=["frontview"],
    video_skip=1,
    method=method,
    score_fn=score_fn,
    guidance_fn=guidance_fn,
)
summary = _summarize_rollout(method, INIT_STATE_DIR, SEED, rollout, run_output_path)
_save_rollout_artifacts(run_output_path, rollout, summary)

print(f"output dir: {run_output_path}")
print(f"video path: {video_path}")
print(json.dumps(summary, indent=2))

In [ ]:
run_output_root = run_output_path if "run_output_path" in globals() else None
if run_output_root is None:
    base_dir = REPO_ROOT / "outputs/can_rollouts/notebook_junk/single_rollouts"
    candidates = sorted([p for p in base_dir.iterdir() if p.is_dir()])
    if not candidates:
        raise FileNotFoundError(f"No rollout outputs found under: {base_dir}")
    run_output_root = candidates[-1]

npz_path = run_output_root / "rollout_numeric.npz"
if not npz_path.exists():
    raise FileNotFoundError(f"Missing rollout_numeric.npz: {npz_path}")

data = np.load(npz_path)
rzz = np.asarray(data["rzz_mujoco"], dtype=np.float32)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(np.arange(len(rzz)), rzz, color="#1f77b4", lw=2.0)
ax.set_title(f"Can Rzz vs timestep\n{run_output_root.name}")
ax.set_xlabel("timestep")
ax.set_ylabel("can Rzz")
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

print(f"loaded: {npz_path}")
print(f"len={len(rzz)} | min={rzz.min():.6f} | mean={rzz.mean():.6f} | max={rzz.max():.6f}")

In [ ]:
# Baseline comparison from original 50-seed run: DP base vs Guidance λ=50

comparison_run_root = Path("/scratch/gilbreth/zoellner/diffusion_rollouts/can_rollouts/comparison/20260407_210830_v1_s1_50")
env_name = INIT_STATE_DIR.name
seed = SEED

method_to_label = {
    "dp_base": "DP base",
    "dp_guidance_l50": "Guidance λ=50",
}
method_to_color = {
    "dp_base": "#1f77b4",
    "dp_guidance_l50": "#d62728",
}

seed_to_rzz = {}
for method_slug in ["dp_base", "dp_guidance_l50"]:
    npz_path = comparison_run_root / method_slug / f"{env_name}__seed_{seed:02d}" / "rollout_numeric.npz"
    if not npz_path.exists():
        raise FileNotFoundError(f"Missing rollout file: {npz_path}")
    data = np.load(npz_path)
    seed_to_rzz[method_slug] = np.asarray(data["rzz_mujoco"], dtype=np.float32)

fig, ax = plt.subplots(figsize=(12, 4))
for method_slug in ["dp_base", "dp_guidance_l50"]:
    rzz = seed_to_rzz[method_slug]
    ax.plot(
        np.arange(len(rzz)),
        rzz,
        lw=2.0,
        color=method_to_color[method_slug],
        label=method_to_label[method_slug],
    )

ax.set_title(
    f"Original run comparison (seed {seed}, {env_name})\n"
    f"{comparison_run_root.name}: DP base vs Guidance λ=50"
)
ax.set_xlabel("timestep")
ax.set_ylabel("can Rzz")
ax.set_ylim(0.9, 1.002)
ax.grid(alpha=0.25)
ax.legend(frameon=True)
plt.tight_layout()
plt.show()

print(f"run root: {comparison_run_root}")
for method_slug in ["dp_base", "dp_guidance_l50"]:
    rzz = seed_to_rzz[method_slug]
    print(f"{method_to_label[method_slug]} | len={len(rzz)} | min={rzz.min():.6f} | mean={rzz.mean():.6f} | max={rzz.max():.6f}")

In [ ]:
# Guided-only comparison sweep: min-EEF-rzz guidance for one init state, seeds 1..50, lambdas 20..70


import time
from tqdm.auto import tqdm

GUIDANCE_SCALES = [20, 30, 40, 50, 60, 70]
SEEDS = list(range(1, 51))
RUN_ROOT = _ensure_dir(REPO_ROOT / "outputs/can_rollouts/comparison/min_eef_rzz_guidance")

# Reuse currently loaded init state when available; otherwise load from disk.
init_state_for_batch = init_state if "init_state" in globals() else load_init_state_json(str(INIT_STATE_PATH))

print(f"run root: {RUN_ROOT}")
print(f"init env: {INIT_STATE_DIR.name}")
print(f"seeds: {SEEDS[0]}..{SEEDS[-1]} ({len(SEEDS)} total)")
print(f"guidance scales: {GUIDANCE_SCALES}")

jobs = [(lam, seed) for lam in GUIDANCE_SCALES for seed in SEEDS]
total_jobs = len(jobs)
durations_s = []

pbar = tqdm(jobs, total=total_jobs, desc="Guided sweep", unit="rollout")
for job_idx, (lam, seed) in enumerate(pbar, start=1):
    method = MethodSpec(
        slug=f"dp_guidance_l{int(lam)}",
        label=f"Guidance λ={int(lam)} (min EEF rzz)",
        kind="guided",
        guidance_scale=float(lam),
    )
    method_root = _ensure_dir(RUN_ROOT / method.slug)
    combo_dir = _ensure_dir(method_root / f"{INIT_STATE_DIR.name}__seed_{seed:02d}")
    video_path = combo_dir / "video.mp4"

    t0 = time.perf_counter()
    rollout = _run_rollout_episode(
        env=env,
        policy=policy,
        init_state=init_state_for_batch,
        seed=seed,
        horizon=HORIZON,
        video_path=video_path,
        camera_names=["frontview"],
        video_skip=1,
        method=method,
        score_fn=score_fn,
        guidance_fn=guidance_fn,
    )
    summary = _summarize_rollout(method, INIT_STATE_DIR, seed, rollout, combo_dir)
    _save_rollout_artifacts(combo_dir, rollout, summary)

    dt = time.perf_counter() - t0
    durations_s.append(dt)
    mean_dt = float(np.mean(durations_s))
    remaining = total_jobs - job_idx
    eta_min = (remaining * mean_dt) / 60.0
    pbar.set_postfix({
        "lam": int(lam),
        "seed": seed,
        "last_s": f"{dt:.1f}",
        "avg_s": f"{mean_dt:.1f}",
        "eta_min": f"{eta_min:.1f}",
    })
    print(f"[{job_idx}/{total_jobs}] {method.slug} seed={seed} | {dt:.2f}s")

print("Done.")
print(f"Saved guided-only sweep to: {RUN_ROOT}")

In [6]:
# SNR k=32 sweep with +EEF_Rzz objective (opposite direction from -EEF).
import time
from tqdm.auto import tqdm

MAX_EEF_RUN_ROOT = _ensure_dir(REPO_ROOT / "outputs/can_rollouts/comparison/max_eef_rzz_guidance")
SNR_K = 32
SEEDS = list(range(48, 51))

# Reuse loaded init state and world model components from previous cells.
if "predictor" not in globals() or "stats" not in globals() or "device" not in globals():
    raise RuntimeError("Run Cell 5 first to load env / policy / world model.")

state_mean = torch.tensor(stats["state_mean"], device=device, dtype=torch.float32).unsqueeze(0)
state_std = torch.tensor(stats["state_std"], device=device, dtype=torch.float32).unsqueeze(0)
action_mean = torch.tensor(stats["action_mean"], device=device, dtype=torch.float32).unsqueeze(0)
action_std = torch.tensor(stats["action_std"], device=device, dtype=torch.float32).unsqueeze(0)
delta_mean = torch.tensor(stats["delta_mean"], device=device, dtype=torch.float32).unsqueeze(0)
delta_std = torch.tensor(stats["delta_std"], device=device, dtype=torch.float32).unsqueeze(0)


def score_fn_plus_eef_rzz_from_state(state_now: np.ndarray, actions: torch.Tensor) -> torch.Tensor:
    # NOTE: sample-and-rank selects argmax(score), so +EEF_Rzz means we rank toward larger EEF_Rzz.
    with torch.no_grad():
        state = torch.as_tensor(state_now, device=device, dtype=torch.float32).unsqueeze(0)
        state = state.expand(actions.shape[0], -1).contiguous()

        horizon = min(int(GUIDANCE_ROLLOUT_STEPS), int(actions.shape[1]))
        eef_rzz_traj = []

        for step_idx in range(horizon):
            action_t = actions[:, step_idx, :]
            state_n = (state - state_mean) / state_std
            action_n = (action_t - action_mean) / action_std
            delta_n = predictor(state_n, action_n)
            delta = delta_n * delta_std + delta_mean
            state = state + delta
            eef_rzz_traj.append(_rotation_6d_to_rzz_torch(state[:, 21:27]))

        eef_rzz_stack = torch.stack(eef_rzz_traj, dim=1)
        return eef_rzz_stack.mean(dim=1)


method = MethodSpec(
    slug=f"snr_k{SNR_K}",
    label=f"SNR k={SNR_K} (+EEF_Rzz objective)",
    kind="sample_and_rank",
    rank_k=SNR_K,
    rank_recompute_interval=8,
    rank_reinject_horizon=8,
)

method_root = _ensure_dir(MAX_EEF_RUN_ROOT / method.slug)
rows_path = method_root / "rollout_rows.jsonl"

init_state_for_batch = init_state if "init_state" in globals() else load_init_state_json(str(INIT_STATE_PATH))
rows = []
durations_s = []

print(f"run root: {MAX_EEF_RUN_ROOT}")
print(f"method: {method.slug}")
print(f"objective: +EEF_Rzz over world-model rollout horizon={GUIDANCE_ROLLOUT_STEPS} (argmax ranking)")
print(f"init env: {INIT_STATE_DIR.name}")
print(f"seeds: {SEEDS[0]}..{SEEDS[-1]} ({len(SEEDS)} total)")

pbar = tqdm(SEEDS, total=len(SEEDS), desc="SNR k32 (+EEF)", unit="rollout")
for idx, seed in enumerate(pbar, start=1):
    combo_dir = _ensure_dir(method_root / f"{INIT_STATE_DIR.name}__seed_{seed:02d}")
    video_path = combo_dir / "video.mp4"

    t0 = time.perf_counter()
    rollout = _run_rollout_episode(
        env=env,
        policy=policy,
        init_state=init_state_for_batch,
        seed=seed,
        horizon=HORIZON,
        video_path=video_path,
        camera_names=["frontview"],
        video_skip=1,
        method=method,
        score_fn=score_fn_plus_eef_rzz_from_state,
        guidance_fn=guidance_fn,
    )
    summary = _summarize_rollout(method, INIT_STATE_DIR, seed, rollout, combo_dir)
    _save_rollout_artifacts(combo_dir, rollout, summary)

    row = {
        "method_slug": method.slug,
        "method_label": method.label,
        "init_name": INIT_STATE_DIR.name,
        "seed": int(seed),
        "success": bool(summary.get("success", False)),
        "total_reward": float(summary.get("total_reward", 0.0)),
        "num_steps": int(summary.get("num_steps", 0)),
        "rollout_dir": str(combo_dir),
        "video_path": str(video_path),
    }
    rows.append(row)

    dt = time.perf_counter() - t0
    durations_s.append(dt)
    mean_dt = float(np.mean(durations_s))
    remaining = len(SEEDS) - idx
    eta_min = (remaining * mean_dt) / 60.0
    pbar.set_postfix({
        "seed": seed,
        "last_s": f"{dt:.1f}",
        "avg_s": f"{mean_dt:.1f}",
        "eta_min": f"{eta_min:.1f}",
    })
    print(f"[{idx}/{len(SEEDS)}] {method.slug} seed={seed} | {dt:.2f}s")

with rows_path.open("w", encoding="utf-8") as f:
    for row in rows:
        f.write(json.dumps(row) + "\n")

success_rate = 100.0 * float(np.mean([r["success"] for r in rows])) if rows else float("nan")
print("Done.")
print(f"Saved SNR +EEF sweep to: {method_root}")
print(f"Wrote rows to: {rows_path}")
print(f"Success rate: {success_rate:.1f}%")

run root: /home/zoellner/src/guided-diffusion/outputs/can_rollouts/comparison/max_eef_rzz_guidance
method: snr_k32
objective: +EEF_Rzz over world-model rollout horizon=8 (argmax ranking)
init env: n16h300r60g1bYfYv1
seeds: 48..50 (3 total)


SNR k32 (+EEF):   0%|                                                                                                        | 0/3 [00:00<?, ?rollout/s]

ObservationKeyToModalityDict: robot0_joint_pos not found, adding robot0_joint_pos to mapping with assumed low_dim modality!
ObservationKeyToModalityDict: robot0_joint_pos_cos not found, adding robot0_joint_pos_cos to mapping with assumed low_dim modality!
ObservationKeyToModalityDict: robot0_joint_pos_sin not found, adding robot0_joint_pos_sin to mapping with assumed low_dim modality!
ObservationKeyToModalityDict: robot0_joint_vel not found, adding robot0_joint_vel to mapping with assumed low_dim modality!
ObservationKeyToModalityDict: robot0_eef_quat_site not found, adding robot0_eef_quat_site to mapping with assumed low_dim modality!
ObservationKeyToModalityDict: robot0_gripper_qvel not found, adding robot0_gripper_qvel to mapping with assumed low_dim modality!
ObservationKeyToModalityDict: lang_emb not found, adding lang_emb to mapping with assumed low_dim modality!
ObservationKeyToModalityDict: timesteps not found, adding timesteps to mapping with assumed low_dim modality!
Observat

Set the action queue! Current lenght: 0
Lenght after action queue insertions: 8


Set the action queue! Current lenght: 0
Lenght after action queue insertions: 8


...

Set the action queue! Current lenght: 0
Lenght after action queue insertions: 8


Set the action queue! Current lenght: 0
Lenght after action queue insertions: 8


Set the action queue! Current lenght: 0
Lenght after action queue insertions: 8


SNR k32 (+EEF): 100%|█████████████████████████████████████████████████| 3/3 [02:42<00:00, 54.25s/rollout, seed=50, last_s=43.9, avg_s=54.2, eta_min=0.0]

[3/3] snr_k32 seed=50 | 43.87s
Done.
Saved SNR +EEF sweep to: /home/zoellner/src/guided-diffusion/outputs/can_rollouts/comparison/max_eef_rzz_guidance/snr_k32
Wrote rows to: /home/zoellner/src/guided-diffusion/outputs/can_rollouts/comparison/max_eef_rzz_guidance/snr_k32/rollout_rows.jsonl
Success rate: 100.0%
